In [1]:
import pandas as pd
import openpyxl
import yaml
import os

In [2]:
# --- Input parameters: pathway names to look up ---
pathways = ["strategy_NDC", "strategy_LEP", "strategy_Conditional"]



# Load the Excel workbook (data_only=True reads computed values instead of formulas)
wb = openpyxl.load_workbook(
    "../scenario_mapping/ssp_libya_transformation_cw_20260309.xlsx",
    data_only=True,
)

# --- Step 1: Build the base dictionary from the "yaml" sheet ---
# The "yaml" sheet has 4 columns:
#   A: subsector
#   B: transformation_code
#   C: transformation_name
#   D: transformation_yaml_name

ws_yaml = wb["yaml"]
rows_yaml = list(ws_yaml.iter_rows(min_row=1, values_only=True))

# First row contains column headers
headers_yaml = rows_yaml[0]

# Build a dictionary keyed by transformation_code (column B, index 1).
# Each value is a sub-dict of the remaining columns (subsector, transformation_name,
# transformation_yaml_name), skipping the key column itself (i != 1).
transformations = {
    row[1]: {h: row[i] for i, h in enumerate(headers_yaml) if i != 1}
    for row in rows_yaml[1:]
}

# --- Step 2: Enrich with transformation_description from the "main" sheet ---
# The "main" sheet contains additional metadata. We only need two columns:
#   Column index 2: transformation_code  (used to match keys in our dict)
#   Column index 4: transformation_description (the new value to add)

ws_main = wb["main"]
# Skip the header row (min_row=2)
rows_main = list(ws_main.iter_rows(min_row=2, values_only=True))

for row in rows_main:
    code = row[2]   # transformation_code — matches keys in `transformations`
    desc = row[4]   # transformation_description — long-form text describing the transformation

    # Only add the description if the code exists in our base dictionary,
    # since the main sheet may contain rows not present in the yaml sheet.
    if code in transformations:
        transformations[code]["transformation_description"] = desc


In [3]:
# --- Step 3: Helper to extract magnitude/parameters from a YAML file ---
transformations_dir = "../transformations"
SKIP_PARAMS = {"vec_implementation_ramp", "return_pathways", "return_prevalence_dict"}
def extract_parameters(yaml_path):
    """Extract the magnitude or relevant parameters from a transformation YAML.
    Returns the value of 'magnitude' if present; otherwise returns a dict
    of all parameters excluding internal/ramp keys.  Returns None when the
    file does not exist or has no usable parameters.
    """
    if not os.path.isfile(yaml_path):
        return None
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    params = data.get("parameters", {})
    if not params:
        return None
    if "magnitude" in params:
        return params["magnitude"]
    # No simple 'magnitude' — collect every meaningful parameter
    relevant = {k: v for k, v in params.items() if k not in SKIP_PARAMS}
    if not relevant:
        return None
    if len(relevant) == 1:
        return list(relevant.values())[0]
    return relevant
# --- Step 4: For each transformation, get base + pathway parameters ---
for code, info in transformations.items():
    base_yaml = info["transformation_yaml_name"]
    base_stem = base_yaml.replace(".yaml", "")
    base_path = os.path.join(transformations_dir, base_yaml)
    # Base (default) magnitude
    info["magnitude_base"] = extract_parameters(base_path)
    # One column per pathway
    for pathway in pathways:
        pathway_path = os.path.join(transformations_dir, f"{base_stem}_{pathway}.yaml")
        info[f"magnitude_{pathway}"] = extract_parameters(pathway_path)
# --- Step 5: Build the output DataFrame ---
df = pd.DataFrame.from_dict(transformations, orient="index")
df.index.name = "transformation_code"
df = df.reset_index()
df

,transformation_code,subsector,transformation_name,transformation_yaml_name,transformation_description,magnitude_base,magnitude_strategy_NDC,magnitude_strategy_LEP,magnitude_strategy_Conditional
0,TX:AGRC:DEC_CH4_RICE,AGRC,Improve rice management,transformation_agrc_dec_ch4_rice.yaml,Many practices can reduce emissions associated...,0.45,None,None,None
1,TX:AGRC:DEC_EXPORTS,AGRC,Decrease Exports,transformation_agrc_dec_exports.yaml,Decrease agricultural exports by some percenta...,0.5,None,None,None
2,TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN,AGRC,Reduce supply chain losses,transformation_agrc_dec_losses_supply_chain.yaml,Reduce waste food waste in the agricultural (c...,0.3,None,None,None
3,TX:AGRC:INC_CONSERVATION_AGRICULTURE,AGRC,Expand conservation agriculture,transformation_agrc_inc_conservation_agricultu...,Conservation agriculture is the term given to ...,"{'dict_categories_to_magnitude': None, 'magnit...",None,None,None
4,TX:AGRC:INC_PRODUCTIVITY,AGRC,Improve crop productivity,transformation_agrc_inc_productivity.yaml,Apply a fractional increase to crop yield fact...,0.2,None,None,None
...,...,...,...,...,...,...,...,...,...
57,TX:WASO:INC_ENERGY_FROM_BIOGAS,WASO,Biogas for energy production,transformation_waso_inc_energy_from_biogas.yaml,Increase the fraction of biogas that is collec...,0.85,0.17,0.085,0.17
58,TX:WASO:INC_ENERGY_FROM_INCINERATION,WASO,Incineration for energy production,transformation_waso_inc_energy_from_incinerati...,Solid waste can be incinerated and used for en...,0.85,0.17,0.085,0.17
59,TX:WASO:INC_LANDFILLING,WASO,Increase landfilling,transformation_waso_inc_landfilling.yaml,Increase fraction of waste that is otherwise n...,1.0,None,None,None
60,TX:WASO:INC_RECYCLING,WASO,Increase recycling,transformation_waso_inc_recycling.yaml,Recycling can reduce emissions by reducing ana...,0.95,None,None,None


In [4]:
df.to_csv("../transformations_description/transformations_description.csv", index=False)

In [5]:
import json
from transformation_templates_v2 import PathwayConfig, build_table_rows

# --- Libya pathway configuration ---
# Strategy names must match the suffixes used in the YAML files
# and the 'pathways' list defined in Cell 1 above.
cfg = PathwayConfig(
    bau="strategy_NDC",
    unconditional="strategy_LEP",
    conditional="strategy_Conditional",
    bau_label="BAU",
    unconditional_label="Unconditional",
    conditional_label="Conditional",
)

rows = build_table_rows(df, config=cfg)
print(f"{len(rows)} transformations loaded")


62 transformations loaded


In [6]:
# --- Export annex_data.json for the Word table generator ---
output_json = "../transformations_description/annex_data.json"
with open(output_json, 'w') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f"Exported → {output_json}")

Exported → ../transformations_description/annex_data.json


In [7]:
# --- Generate the Word annex table ---
# Requires: node + docx npm package (npm install -g docx)
# generate_annex_table.js must be in the same folder as this notebook.
import subprocess
result = subprocess.run(
    ["node", "generate_annex_table.js"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

Done → annex_transformations.docx

